## Adversarial Prompt Detection

This notebook implements a pipeline for detecting adversarial prompts using Bayesian uncertainty quantification over LoRA parameters.

### Pipeline

1. **Data Loading**: Load safe, benign, and harmful prompts for training and testing
2. **Model Setup**: Initialize a language model with LoRA adapters
3. **Fine-tuning**: Train the LoRA adapters on safe prompts
4. **Laplace Approximation**: Compute posterior distribution over LoRA parameters using diagonal Fisher information
5. **Uncertainty Quantification**: Compute entropy and credal set metrics to distinguish adversarial from safe prompts

### Hypothesis

Adversarial prompts should exhibit higher uncertainty (entropy) and wider credal sets compared to safe prompts, enabling their detection.


In [ ]:
import torch 
import numpy as np
from scipy import stats
import src.constants as constants

from transformers import AutoModelForCausalLM, AutoTokenizer

from src.data_utils import create_dataloader, load_training_and_test_data

from src.training import setup_model_and_lora, train_lora

from src.laplace import collect_laplace_data, compute_diagonal_fisher
from src.uncertainty import compute_predictive_credal_sets, compute_predictive_entropy


#### Load data

In [ ]:
data = load_training_and_test_data(
    n_safe_train=constants.N_SAFE_TRAIN,
    n_benign_train=constants.N_BENIGN_TRAIN,
    n_test_per_category=constants.N_TEST_PER_CATEGORY
)

train_prompts = data['train_prompts']
safe_test = data['safe_test']
harmful_test = data['harmful_test']

#### Load model and tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(constants.MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    constants.MODEL_NAME,
    torch_dtype=torch.float16 if constants.DEVICE == "cuda" else torch.float32,
    device_map="auto"
)

In [ ]:
model, tokenizer = setup_model_and_lora(constants.MODEL_NAME, constants.DEVICE, lora_rank=constants.LORA_RANK)

#### Fine-tune model with LoRA

In [ ]:
train_loader = create_dataloader(
    train_prompts, 
    tokenizer, 
    max_length=constants.MAX_LENGTH, 
    batch_size=constants.BATCH_SIZE, 
    shuffle=True
)


In [ ]:
model = train_lora(model, train_loader, epochs=constants.EPOCHS, lr=constants.LEARNING_RATE, device=constants.DEVICE)

In [ ]:
model_save_path = "models/fine_tuned_lora_model"
print(f"\nSaving fine-tuned model to {model_save_path}...")
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

#### Compute Laplace Approximation to get Posterior Distribution Over Lora Model Parameters

In [ ]:
# Collect limited data for Laplace
laplace_data = collect_laplace_data(
    model,
    train_loader,
    max_batches=constants.MAX_LAPLACE_BATCHES
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Compute diagonal Fisher (Laplace approximation)
fisher_diag, lora_params = compute_diagonal_fisher(
    model,
    laplace_data,
    device=constants.DEVICE
)

In [ ]:
model = model.float()

#### Compute Entropy for Safe and Adversarial Prompts

In [ ]:
# Compute entropy for safe prompts
print("\nProcessing safe prompts...")
safe_entropies = compute_predictive_entropy(
    model=model,
    prompts=safe_test,
    tokenizer=tokenizer,
    fisher_diag=fisher_diag,
    n_samples=constants.N_POSTERIOR_SAMPLES, 
    temperature=0.05,
    device=constants.DEVICE
)

# Clear cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Compute entropy for adversarial prompts
print("\nProcessing adversarial prompts...")
adv_entropies = compute_predictive_entropy(
    model=model,
    prompts=harmful_test,
    tokenizer=tokenizer,
    fisher_diag=fisher_diag,
    n_samples=constants.N_POSTERIOR_SAMPLES,
    temperature=0.05,
    device=constants.DEVICE
)

In [ ]:
safe_mean = np.mean(safe_entropies)
safe_std = np.std(safe_entropies)
adv_mean = np.mean(adv_entropies)
adv_std = np.std(adv_entropies)

print(f"\nSafe Prompts:")
print(f"  Mean entropy: {safe_mean:.4f} ± {safe_std:.4f}")
print(f"  Min: {np.min(safe_entropies):.4f}, Max: {np.max(safe_entropies):.4f}")

print(f"\nAdversarial Prompts:")
print(f"  Mean entropy: {adv_mean:.4f} ± {adv_std:.4f}")
print(f"  Min: {np.min(adv_entropies):.4f}, Max: {np.max(adv_entropies):.4f}")

In [ ]:
t_stat, p_value = stats.ttest_ind(adv_entropies, safe_entropies)
print(f"\nStatistical Test:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4e}")

if p_value < 0.05:
    print(f"Significant difference (p < 0.05)")
    if adv_mean > safe_mean:
        print(f"Adversarial prompts have HIGHER entropy (supports hypothesis)")
    else:
        print(f"Adversarial prompts have LOWER entropy (unexpected)")
else:
    print(f" No significant difference (p ≥ 0.05)")


#### Compute Credal Set Metrics

##### Credal Set Metrics on Full Vocabulary Size 

In [ ]:
safe_credal_results = compute_predictive_credal_sets(model, safe_test, tokenizer, fisher_diag,
                                   n_samples=20, temperature=0.05, 
                                   top_k=None, device=constants.DEVICE)

adv_credal_results = compute_predictive_credal_sets(model, harmful_test, tokenizer, fisher_diag,
                                   n_samples=20, temperature=0.05, 
                                   top_k=None, device=constants.DEVICE)

##### Credal Set Metrics on Budgeted Vocabulary Size (Top-K)

In [ ]:
budgeted_safe_credal_results = compute_predictive_credal_sets(model, safe_test, tokenizer, fisher_diag,
                                   n_samples=20, temperature=0.05, 
                                   top_k=100, device="cuda")

budgeted_adv_credal_results = compute_predictive_credal_sets(model, harmful_test, tokenizer, fisher_diag,
                                   n_samples=20, temperature=0.05, 
                                   top_k=100, device="cuda")